# MNT02 미션 05 — 적정 안전재고량 산출 및 결품 위험 지수(Stockout Risk) 엔진 개발

**Student 학습용** / CMS 편성 4시간 / Level 3

## 시작 전 꼭 읽으세요

이번 활동은 **교육용 가상 부품 데이터**를 분석하는 실습입니다. 코딩이 처음이어도 안내와 TODO를 따라 한 단계씩 진행하세요.

1. 아래 **준비 셀**을 실행합니다. 필요한 라이브러리는 실습 환경에 설치되어 있어야 합니다.
2. 미션의 입력 자료·핵심 함수를 확인합니다. 학생용 `TODO` 부분을 직접 작성하거나 허용된 AI에 도움을 요청합니다.
3. 코드를 실행하고 실제 출력·오류를 기록합니다. 실행이 실패했다면 실패 내용을 그대로 남기고 수정 후 다시 실행하세요.
4. AI에게 보낸 **실제 질문과 받은 답변 원문**을 작성 기록에 붙여넣고, 코드·출력과 비교한 수정 이유를 작성합니다.
5. 마지막 기록 셀을 실행해서 JSON과 검토용 ZIP을 만들고 내려받습니다. 파일 생성만으로 학습 PASS나 사람의 승인이 되지 않습니다.

**인터넷/폐쇄망:** Colab과 외부 AI는 접속이 허용된 환경에서만 사용합니다. 폐쇄망에서는 사전 설치된 로컬 Jupyter 등 허용된 환경이 필요하며, 외부 AI 사용이 불가능하면 '미실시'로 기록하세요.

**보안:** 실제 부대·조달·업체·정비 자료를 입력하거나 AI로 실제 발주를 승인하지 마세요.


**실습 전 필수:** ① 환경 설정 셀에서 `MNT02_실습프로젝트.zip` 또는 다른 이름의 실습자료 `.zip`을 업로드하고 압축 해제 완료를 확인하세요. 압축 업로드가 끝나야 핵심 실습 코드가 실행됩니다. 환경 설정 및 통합 저장 코드셀은 기본 숨김이며 실행 버튼만 사용하면 됩니다.


In [ ]:
# @title ① 필수 실습자료 ZIP 업로드 및 환경 준비 (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 권장 자료명: MNT02_실습프로젝트.zip / 실제 업로드 파일 이름은 자유입니다.
# Colab에서는 한 개의 ZIP을 선택합니다. 로컬 Jupyter에서는 업로드 위젯을 사용합니다.
WORKSPACE = (Path.cwd() / 'MNT02_실습프로젝트_작업공간').resolve()
MNT02_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global MNT02_PROJECT_READY, MNT02_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        members = [m for m in archive.infolist() if not m.is_dir()]
        if not members:
            raise ValueError('ZIP 안에 실습자료 파일이 없습니다.')
        if len(members) > 15000 or sum(m.file_size for m in members) > 800*1024*1024:
            raise ValueError('ZIP이 너무 크거나 파일 개수가 너무 많습니다.')
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('wb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    MNT02_PROJECT_DIR = WORKSPACE
    MNT02_PROJECT_READY = True
    print('✅ 실습자료 압축 해제 완료:', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

print('【필수】실습 시작 전 프로젝트 실습자료 ZIP 파일을 업로드해야 합니다.')
print('권장 프로젝트 자료: MNT02_실습프로젝트.zip (파일명 변경 가능)')
print('ZIP 파일명은 자유입니다. 확장자가 .zip인 파일을 선택하세요.')
try:
    import google.colab
    from google.colab import files as colab_files
    print('① [파일 선택]에서 실습자료 .zip 파일을 한 개 선택하세요.')
    incoming = colab_files.upload()
    zip_items = [(name, data) for name, data in incoming.items() if name.lower().endswith('.zip')]
    if len(zip_items) != 1 or len(incoming) != 1:
        raise RuntimeError('ZIP 파일 한 개만 업로드해 주세요. 이 셀을 다시 실행하세요.')
    selected_name, selected_data = zip_items[0]
    print('선택한 ZIP:', selected_name)
    _prepare_project_zip(selected_data)
except ImportError:
    # 로컬 Jupyter: 파일명이 달라도 .zip 확장자이면 업로드할 수 있습니다.
    import ipywidgets as widgets
    from IPython.display import display
    uploader = widgets.FileUpload(accept='.zip', multiple=False, description='실습 ZIP 업로드')
    upload_log = widgets.Output()
    def _on_upload(change):
        with upload_log:
            upload_log.clear_output()
            try:
                values = uploader.value
                items = list(values.values()) if isinstance(values, dict) else list(values)
                if len(items) != 1:
                    print('❌ ZIP 파일 한 개를 선택해 주세요.')
                    return
                selected = items[0]
                filename = selected['name']
                if not filename.lower().endswith('.zip'):
                    print('❌ .zip 확장자 파일만 선택할 수 있습니다.')
                    return
                print('선택한 ZIP:', filename)
                _prepare_project_zip(bytes(selected['content']))
            except Exception as exc:
                print('❌ 업로드/압축 해제 실패:', exc)
    uploader.observe(_on_upload, names='value')
    display(Markdown('**필수:** 프로젝트 실습자료가 들어 있는 `.zip` 파일을 선택하세요. 파일명은 자유이며, **압축 해제 완료** 문구를 확인한 뒤 다음 셀로 이동합니다.'))
    display(uploader, upload_log)
    print('⏳ 실습 ZIP 업로드 대기 중입니다.')
print('실행 환경: Python', sys.version.split()[0])


## 상황: 어떤 문제를 해결하나요?

미션 개요 앙상블 수요 예측 모델을 바탕으로 품목별 적정 안전재고량과 재주문점을 산출하고, 단종 상태 및 공급망 위험 요소를 반영한 결품 위험 지수(Stockout Risk) 엔진을 개발하는 실습입니다. 배경 및 필요성 정확한 수요 예측을 수행하더라도 예상치 못한 수요 변동이나 긴 조달 리드타임으로 인해 결품이 발생할 수 있으므로, 통계적 근거에 기반한 안전재고와 재주문점 설정이 필수적입니다. 특히 생산 중단(단종)이나 단종 예고, 단일 공급업체, 대체품 부재 등의 구조적 위험 요소를 종합하여 객관적인 결품 위험 지수를 산출해야 최우선 조달 품목을 선별할 수 있습니다. 핵심 과업 수요 예측 앙상블 고도화: 포아송 그래디언트 부스팅과 랜덤포레스트 모델을 앙상블하고, 학습 기간 내 검증 구간을 통해 최적의 가중치를 도출합니다. Hold-out 6개월 구간에서 기준 모델(MA3) 대비 MAE 및 RMSE 개선율과 결정계수($R^2$)를 검증합니다. 안전재고 및 재주문점 산출: 예측 오차를 활용하여 서비스 수준 95% 기준의 적정 안전재고량과 재주문점을 계산합니다. 결품 위험 지수 엔진 개발: 리드타임 동안의 결품 확률에 단종 상태, 단일 공급업체 유무, 대체품 부재 여부를 결합하여 종합 위험 지수를 산출하고 위험 등급(경보·주의·관찰)을 분류합니다. 검증 및 산출물 검증 기준: 기준 모델 대비 MAE/RMSE 20% 이상 개선, 결정계수($R^2$) 0.85 이상 달성, 품목별 안전재고·재주문점 및 위험 등급(경보·주의·관찰) 산출 완료. 주요 산출물: src/models/mnt02_advanced_model.py, 안전재고 및 결품 위험 지수표.

## 이번 미션에서 완성할 것

포아송 그래디언트 부스팅과 랜덤포레스트를 앙상블하고 가중치는 학습 기간 안의 검증 구간으로 고른다. Hold-out 6개월에서 기준 모델(MA3) 대비 MAE/RMSE 개선율과 R2를 계산한다. 예측 오차로 서비스 수준 95% 안전재고와 재주문점을 구하고, 리드타임 동안의 결품 확률에 단종 상태·단일 공급업체·대체품 부재를 결합해 위험 지수와 등급을 산출한다. 핵심 구현 계약 src/models/mnt02_advanced_model.py / stock_policy: items,mu,sigma,z,review → 품목별 mu/sigma/L/ss/rop DataFrame. 제공 리드타임+검토주기로 안전재고·재주문점을 계산한다. 정규분포 가정이며 실제 서비스 수준을 보장하지 않는다. 추가 변경·검증 앙상블 수요 예측으로 품목별 적정 안전재고·재주문점을 산출하고 단종 위험을 반영한 결품 위험 지수(Stockout Risk) 엔진을 개발한다. 단종예고 가중치 0.5→0.8 변경 전후의 위험 품목 수·점수 변화를 비교하고 모델 성능과 위험 지수를 분리한다. 핵심구현 후첫성공실행을보존하고추가변경·재실행·실제AI원문·근거검토3건·보완·최종판단을 작성합니다.

**제출 경로:** `src/models/mnt02_advanced_model.py`, `reports/mnt02_model_metrics.md`, `forms/MNT02_M05_forms.md`



---

## 🎯 미션 05. 적정 안전재고량 산출 및 결품 위험 지수(Stockout Risk) 엔진 개발
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

적정 안전재고량 산출 및 결품 위험 지수 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **앙상블 수요 예측으로 품목별 적정 안전재고·재주문점을 산출하고 단종 위험을 반영한 결품 위험 지수(Stockout Risk) 엔진을 개발한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **포아송 그래디언트 부스팅과 랜덤포레스트를 앙상블하고 가중치는 학습 기간 안의 검증 구간으로 고른다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/mnt02_advanced_model.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 비정형/불규칙 수요에 대한 모델의 적응력 및 예측 안정성

#### 미션 PASS 조건

- 정량 기준: 기준 모델 대비 오차율(MAE/RMSE) 20% 이상 개선, R2 Score 0.85 이상


- 진행 기준: 모델 하이퍼파라미터 튜닝 완료 및 최적 모델 가중치 저장


- 검증 방법: Hold-out 테스트 데이터셋 대상 최종 성능 평가


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



## 순서대로 할 활동

### 활동 1. 계획과 사전 판단

원문 목표 및 예측 오차·공급 요인 연계 분석 원본 목표, 기본 제공 코드, 그리고 관련 자료를 면밀히 검토하여 앙상블 수요 예측 결과를 품목별 적정 안전재고 및 재주문점(ROP) 산출 과정에 어떻게 유기적으로 연결할 것인지 기획합니다. 예측 오차, 리드타임(L), 그리고 주요 공급 요인들이 안전재고 계산 수식에 미치는 영향을 파악하고, 단종 위험을 반영한 결품 위험 지수(Stockout Risk) 엔진의 설계 방향을 수립합니다. AI 위임 범위 및 검증 계획 설정 안전재고 및 재주문점 산출 로직 구현 시 AI에 위임할 코드 작성 범위와 활용할 자료를 구체적으로 확정합니다. 입력 데이터의 기간, 단위, 사용 가능 시점과 stock_policy 함수의 입출력 계약(Contract)을 명확히 정의하고, 본인의 사전 판단 및 검증 계획을 철저히 준비합니다.

**확인할 사항:** 이 활동의 확인 근거 M05 계획: 앙상블 수요 예측으로 품목별 적정 안전재고·재주문점을 산출하고 단종 위험을 반영한 결품 위험 지수(Stockout Risk) 엔진을 개발한다.에 필요한 입력·기간·단위·사용 가능 시점과 stock_policy의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가? 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2. AI와 코드 변경

핵심 함수(stock_policy) 구현 및 첫 실행 보존 items, mu, sigma, z, review 인자를 받아 품목별 평균 수요량(mu), 표준편차(sigma), 리드타임($L$), 안전재고(ss), 재주문점(rop)을 포함한 DataFrame을 산출하는 핵심 함수 stock_policy를 구현하고 저장한 뒤, 첫 성공 실행 결과를 온전히 보존합니다. 제공된 리드타임과 검토 주기를 바탕으로 안전재고와 재주문점을 계산하며, 정규분포 가정을 기반으로 하므로 실제 서비스 수준을 완벽히 보장하지는 않음을 명시합니다. AI와 기획 반영 및 변경 과제 수행 앞서 수립한 안전재고 및 재주문점 산출 기획안을 바탕으로 구현된 코드를 검증하고 AI와 함께 고도화 작업을 진행합니다. 변경 과제: 단종 예고 가중치를 0.5에서 0.8로 변경했을 때의 위험 품목 수 및 점수 변화를 비교하고, 일반 수요 모델 성능 지표와 단종 위험 지수를 명확히 분리하여 검증합니다. 실제 코드와 실행 출력 결과물은 모두 안전하게 저장합니다.

**확인할 사항:** 이 활동의 확인 근거 M05 구현: src/models/mnt02_advanced_model.py의 stock_policy를저장하고계약〔items,mu,sigma,z,review → 품목별 mu/sigma/L/ss/rop DataFrame. 제공 리드타임+검토주기로 안전재고·재주문점을 계산한다. 정규분포 가정이며 실제 서비스 수준을 보장하지 않는다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가? 이후Lesson의추가변경을실제코드에반영했는가? 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3. 근거 대조와 재실행

수정 코드 저장 및 재실행 검증 안전재고 및 결품 위험 지수 산출 로직에 반영된 수정 코드를 안전하게 저장하고 스크립트를 재실행합니다. 위험 가중치 변화가 단순한 예측 성능(MAE/RMSE/R2) 개선으로 잘못 해석되지 않았는지 엄격히 구분하여 검증합니다. AI 설명 및 코드 선택 대조 분석 AI가 제시한 설명 및 코드 선택 사항 3건 이상을 실제 실습 자료, 실행 결과 로그와 상호 대조하여 타당성을 엄격히 검토합니다. 보완이 필요한 답변과 수정 사항을 다시 한번 확인하고, 가중치는 t36～41 구간에서 선정하고 t42～47 구간을 성능 평가에 사용했는지, 다음 달 위험 계산 시 당시까지 알려진 과거 오차를 활용했는지 점검합니다. 95% z값은 통계적 분포 가정에 불과하며 실제 실측 충족률을 완벽히 보장하지 않는다는 점과, 0～100 사이의 위험 지수는 보정된 단종 확률이 아님을 명확히 유의합니다.

**확인할 사항:** 이 활동의 확인 근거 M05 검증: [필수] 안전재고·위험 지수 — 기준 모델 대비 MAE/RMSE 20% 이상 개선, R2 0.85 이상, 품목별 안전재고·재주문점과 위험 등급(경보·주의·관찰)을 산출했는가. 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가? 가중치는t36～41에서 고르고t42～47을 성능 평가에 사용합니다. 다음달 위험 계산에는 당시까지 알려진 과거 오차를 사용할 수 있습니다. 95%z는 분포 가정이며 실측 충족률 보장이 아닙니다. 위험 지수 0～100은 보정된 단종 확률이 아닙니다. AI 주장/선택3건을근거로검토·보완·재실행했는가? 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4. 최종 판단과 산출물

종합 산출물 제출 및 상태 구분 원본 평가 기준에 따른 최종 결과물인 src/models/mnt02_advanced_model.py, reports/mnt02_model_metrics.md, forms/MNT02_M05_forms.md의 실제 작성 내용, 최종 측정값, AI 대화 보존 위치, 선택 이유를 상호 연결하여 체계적으로 제출합니다. 미달 항목, 미측정 항목, 문서 미작성 내용, 그리고 실제 승인 대기 상태를 명확히 구분하여 기록합니다. 평가 구간 및 검증 기준 준수 가중치는 t36～41 구간에서 선정하고, t42～47 구간을 성능 평가에 엄격히 사용합니다. 다음 달 위험 계산 시에는 당시까지 알려진 과거 오차만을 활용할 수 있도록 합니다. 95% z값은 통계적 분포 가정에 불과하며 실제 실측 충족률을 완벽히 보장하지 않는다는 점과, 0～100 사이의 위험 지수는 보정된 단종 확률이 아님을 명확히 명시합니다.

**확인할 사항:** 이 활동의 확인 근거 M05 완료: src/models/mnt02_advanced_model.py, reports/mnt02_model_metrics.md, forms/MNT02_M05_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가? 미달·미측정·문서미작성·실제승인대기를정확히표시했는가? 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 이 미션의 AI 활용 예시

- 가상 자료 분석·기획·코드 변경·디버깅·검증 설계·해석
- "교육용 미션 05의 핵심 함수 구현에서 누락되기 쉬운 입력값·경계조건을 알려주세요. 원본 자료에 없는 성능 수치를 만들지 말아 주세요."

AI 답변은 그대로 정답이 아닙니다. **실제 실행 로그와 최소 3건의 주장·선택을 비교**하세요.


## 직접 구현하기

다음 코드의 `TODO`를 완성해 실행합니다. 출력을 확인하지 않았다면 성공했다고 적지 마세요.


In [ ]:
if not globals().get('MNT02_PROJECT_READY', False):
    raise RuntimeError('실습 시작 전 ① 환경 설정 셀에서 프로젝트 실습자료 ZIP을 업로드하세요.')

"""포아송 그래디언트 부스팅·랜덤포레스트 앙상블로 수요 예측을 고도화해 이동평균 기준 모델 대비 MAE/RMSE 개선율과 R2를 검증하고, 예측 오차로 품목별 적정 안전재고·재주문점을 산출한 뒤 리드타임 결품 확률과 단종 상태·단일 공급업체·대체품 부재를 결합한 결품 위험 지수 엔진을 만든다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 야전 정비창 수리부속 청구/출고·장비 가동률 데이터 생성기
# 실제 부대·품목·재고·단가 정보와 무관하다. 품목 40개 x 48개월(가상 2022-01 ~ 2025-12) 월별 자료를 만든다.
# ---------------------------------------------------------------------------
FSC = {"2520": "power_train", "2530": "brake_steer", "2910": "fuel", "2930": "cooling",
       "2940": "filter", "5330": "gasket", "6140": "battery", "5961": "semiconductor"}
EXERCISE = {3, 8, 10}      # 가상 대규모 훈련 월(훈련 계획으로 미리 알 수 있음)
N_MONTHS = 48


def season(fsc, month):
    """품목군별 계절 계수: 축전지는 겨울, 냉각계통은 여름에 수요가 늘어난다."""
    # [TODO : 학생 실습 5] 적정 안전재고량 산출 및 결품 위험 지수(Stockout Risk) 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_items(n=40, seed=2):
    """품목 마스터: NSN(가상), 품목군, 장비 유형, 기준 수요, 단가, 조달 리드타임, 공급업체 수, 생산(단종) 상태."""
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        f = str(rng.choice(list(FSC)))
        status = str(rng.choice(["생산중", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
        rows.append({"item_id": f"P{i + 1:03d}", "nsn": f"{f}-00-{rng.integers(100, 1000)}-{rng.integers(1000, 10000)}",
                     "fsc": f, "group": FSC[f], "equip_type": str(rng.choice(["A", "B", "C"])),
                     "base": float(np.round(rng.lognormal(2.7, 0.8), 1)),
                     "price": int(rng.lognormal(12, 1.0)) // 100 * 100,
                     "lead_time": int(rng.integers(1, 7) if status == "생산중" else rng.integers(4, 13)),
                     "suppliers": int(rng.integers(1, 5)) if status == "생산중" else 1,
                     "status": status, "alt_available": bool(rng.random() < (0.6 if status == "생산중" else 0.3))})
    return pd.DataFrame(rows)


def make_history(items, months=N_MONTHS, seed=3):
    """월별 청구 수요(개), 긴급 불출 건수, 장비 유형별 운용시간·가동률."""
    rng = np.random.default_rng(seed)
    t = np.arange(months)
    moy = t % 12 + 1
    ex = np.isin(moy, list(EXERCISE))
    hours = {e: (100 + 20 * rng.random(months)) * np.where(ex, 1.5, 1.0) for e in "ABC"}
    avail = {e: np.clip(0.93 - 0.12 * (hours[e] - 100) / 80 + rng.normal(0, 0.015, months), 0.6, 0.99) for e in "ABC"}
    rows = []
    for it in items.itertuples():
        trend = 1 + rng.uniform(-0.1, 0.3) * t / months
        s = np.array([season(it.fsc, m) for m in moy])
        lam = it.base * s * trend * hours[it.equip_type] / 110
        d = rng.poisson(lam)
        rows.append(pd.DataFrame({"item_id": it.item_id, "t": t, "ym": [f"{2022 + k // 12}-{k % 12 + 1:02d}" for k in t],
                                  "month": moy, "exercise": ex.astype(int), "op_hours": hours[it.equip_type].round(1),
                                  "availability": avail[it.equip_type].round(3), "demand": d,
                                  "emergency": rng.binomial(d, 0.05)}))
    return pd.concat(rows, ignore_index=True)


def make_stock(items, hist, seed=4):
    """현재(가상 2026-01) 재고: 보유량과 미입고 발주량(조달 중인 물량). 단종 품목은 추가 발주가 어려워 미입고량이 적다."""
    rng = np.random.default_rng(seed)
    recent = hist[hist["t"] >= hist["t"].max() - 5].groupby("item_id")["demand"].mean()
    st = items[["item_id", "lead_time", "status"]].copy()
    st["on_hand"] = [int(round(recent[i] * rng.uniform(0.5, 3.0))) for i in st["item_id"]]
    cover = np.where(st["status"] == "생산중", rng.uniform(0.6, 1.3, len(st)), rng.uniform(0.0, 0.6, len(st)))
    st["on_order"] = [int(round(recent[i] * L * c)) for i, L, c in zip(st["item_id"], st["lead_time"], cover)]
    return st[["item_id", "on_hand", "on_order"]]
# ---------------------------------------------------------------------------

# ---- 수요 예측 공통 함수(파생 피처 + 머신러닝 모델) ----
FEATS = ["lag1", "lag2", "lag3", "lag12", "ma3", "ma6", "std6", "season_idx", "month_sin", "month_cos",
         "exercise", "op_hours", "op_ratio", "item_level", "trend_t"]


def make_features(hist):
    """품목별 과거 수요로 파생 피처를 만든다(모두 예측 시점 이전 정보만 사용)."""
    df = hist.sort_values(["item_id", "t"]).reset_index(drop=True)
    g = df.groupby("item_id")["demand"]
    for k in (1, 2, 3, 12):
        df[f"lag{k}"] = g.shift(k)
    df["ma3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
    df["ma6"] = g.transform(lambda s: s.shift(1).rolling(6).mean())
    df["std6"] = g.transform(lambda s: s.shift(1).rolling(6).std())
    df["item_level"] = g.transform(lambda s: s.shift(1).rolling(12).mean())
    df["season_idx"] = df["lag12"] / (g.transform(lambda s: s.shift(12).rolling(12).mean()) + 1)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["op_ratio"] = df["op_hours"] / df.groupby("item_id")["op_hours"].transform(lambda s: s.shift(1).rolling(12).mean())
    df["trend_t"] = df["t"]
    return df.dropna().reset_index(drop=True)


def new_model():
    """Prophet/LSTM 대신 사용하는 포아송 손실 그래디언트 부스팅(개수 자료에 적합)."""
    # [TODO : 학생 실습 5] 적정 안전재고량 산출 및 결품 위험 지수(Stockout Risk) 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def mape(y, p, min_y=1):
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = y >= min_y
    return float(np.mean(np.abs(y[m] - p[m]) / y[m]))

# ---- 앙상블 수요 예측·안전재고·결품(단종) 위험 지수 공통 함수 ----
STATUS_W = {"생산중": 0.0, "단종예고": 0.5, "단종": 1.0}     # [교육용가상자료] 단종 상태 가중치


def fit_ens(tr):
    """포아송 그래디언트 부스팅(70%) + 랜덤포레스트(30%) 앙상블."""
    h = new_model().fit(tr[FEATS], tr["demand"])
    r = RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=0, n_jobs=1).fit(tr[FEATS], tr["demand"])
    return h, r


def pred_ens(m, X, w=0.7):
    return np.clip(w * m[0].predict(X[FEATS]) + (1 - w) * m[1].predict(X[FEATS]), 0, None)


def next_month(hist, items, t_new=48, op_mult=1.0, ex=0):
    """다음 달(t_new) 예측용 행: 작년 같은 달 운용시간 x 배수, 훈련 여부 지정."""
    last = hist[hist["t"] == t_new - 1].copy()
    prev = hist[hist["t"] == t_new - 12].set_index("item_id")["op_hours"]
    m = (t_new % 12) + 1
    f = last.assign(t=t_new, ym=f"{2022 + t_new // 12}-{m:02d}", month=m, demand=0, exercise=ex,
                    op_hours=last["item_id"].map(prev).to_numpy() * op_mult)
    X = make_features(pd.concat([hist, f], ignore_index=True))
    return X[X["t"] == t_new].set_index("item_id").loc[items["item_id"]].reset_index()


def stock_policy(items, mu, sigma, z=1.65, review=1):
    """안전재고 = z x 월 예측오차 표준편차 x sqrt(리드타임+검토주기), 재주문점 = 월 수요 x (리드타임+검토주기) + 안전재고."""
    L = items["lead_time"].to_numpy() + review
    ss = z * sigma * np.sqrt(L)
    return pd.DataFrame({"item_id": items["item_id"], "mu": mu, "sigma": sigma, "L": L,
                         "ss": np.ceil(ss), "rop": np.ceil(mu * L + ss)})


def risk_index(items, pol, stock):
    """결품 위험 지수(0~100): 리드타임 동안 수요가 가용 재고를 넘을 확률(정규 근사)을 기본으로,
    단종 상태·단일 공급업체·대체품 부재에 가중을 준다(가상 가중치)."""
    st = stock.set_index("item_id").loc[items["item_id"]]
    avail = (st["on_hand"] + st["on_order"]).to_numpy()
    p = 1 - norm.cdf((avail - pol["mu"].to_numpy() * pol["L"].to_numpy()) / (pol["sigma"].to_numpy() * np.sqrt(pol["L"].to_numpy()) + 1e-6))
    sw = items["status"].map(STATUS_W).to_numpy()
    sup = (items["suppliers"].to_numpy() == 1).astype(float)
    alt = (~items["alt_available"].to_numpy()).astype(float)
    score = 100 * np.clip(0.55 * p + 0.25 * sw + 0.10 * sup + 0.10 * alt, 0, 1)
    return pd.DataFrame({"item_id": items["item_id"], "p_short": p, "status_w": sw, "single_sup": sup, "no_alt": alt,
                         "avail": avail, "risk": score.round(1)})

items = make_items()
hist = make_history(items)
df = make_features(hist)
stock = make_stock(items, hist)

# 1) Hold-out(t=42~47) 성능 비교: 기준 모델(MA3) / 단일 모델 / 앙상블. 앙상블 가중치는 t=36~41 검증 구간에서 고른다
inner = fit_ens(df[df["t"] < 36]); vv = df[(df["t"] >= 36) & (df["t"] < 42)]
best_w = min((0.5, 0.6, 0.7, 0.8, 0.9, 1.0), key=lambda w: mean_absolute_error(vv["demand"], pred_ens(inner, vv, w)))
tr, te = df[df["t"] < 42], df[df["t"] >= 42].copy()
ens = fit_ens(tr)
te["ens"], te["hgb"] = pred_ens(ens, te, best_w), np.clip(ens[0].predict(te[FEATS]), 0, None)
rows = {}
for nm, col in [("기준 모델(MA3)", "ma3"), ("단일 포아송 GBM", "hgb"), (f"앙상블(w={best_w})", "ens")]:
    rows[nm] = {"MAE": mean_absolute_error(te["demand"], te[col]), "RMSE": mean_squared_error(te["demand"], te[col]) ** 0.5,
                "R2": r2_score(te["demand"], te[col])}
res = pd.DataFrame(rows).T
print("[Hold-out 6개월(t=42~47), 품목 40개]")
print(res.to_string(float_format=lambda x: f"{x:.3f}"))
b, e = res.iloc[0], res.iloc[2]
im, ir = 1 - e["MAE"] / b["MAE"], 1 - e["RMSE"] / b["RMSE"]
print(f"오차율 개선 MAE {im:.1%}, RMSE {ir:.1%} (목표 20% 이상: {'달성' if min(im, ir) >= 0.2 else '미달'}), R2 {e['R2']:.3f} (목표 0.85 이상: {'달성' if e['R2'] >= 0.85 else '미달'})")
spike = te["exercise"] == 1
print(f"불규칙 수요(훈련 월) MAE: 기준 {mean_absolute_error(te.loc[spike, 'demand'], te.loc[spike, 'ma3']):.2f} -> 앙상블 {mean_absolute_error(te.loc[spike, 'demand'], te.loc[spike, 'ens']):.2f}")

# 2) 안전재고·재주문점: 다음 달 예측과 Hold-out 품목별 오차 표준편차
X = next_month(hist, items)
mu = pred_ens(fit_ens(df), X, best_w)
err = (te["demand"] - te["ens"]).groupby(te["item_id"]).std().reindex(items["item_id"]).to_numpy()
sigma = np.maximum(err, np.sqrt(np.maximum(mu, 1)))          # 오차 표준편차가 포아송 변동보다 작으면 포아송 기준 사용
pol = stock_policy(items, mu, sigma)
risk = risk_index(items, pol, stock)
out = items[["item_id", "group", "status", "lead_time", "suppliers"]].merge(pol, on="item_id").merge(risk, on="item_id")
print(f"\n[다음 달(가상 2026-01) 안전재고 정책] 서비스 수준 95%(z=1.65), 안전재고 합 {out['ss'].sum():.0f}개")
top = out.sort_values("risk", ascending=False).head(8)
show = top[["item_id", "group", "status", "lead_time", "mu", "ss", "rop", "avail", "p_short", "risk"]].copy()
show.columns = ["품목", "품목군", "생산상태", "LT(월)", "월수요", "안전재고", "재주문점", "가용재고", "결품확률", "위험지수"]
print("[결품 위험 지수 상위 8개]")
print(show.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
grade = pd.cut(out["risk"], [-1, 30, 60, 101], labels=["관찰", "주의", "경보"]).value_counts()
print(f"위험 등급: 경보 {grade['경보']}개, 주의 {grade['주의']}개, 관찰 {grade['관찰']}개 / 재주문점 미만 품목 {(out['avail'] < out['rop']).sum()}개")
print("단종·단종예고 품목은 리드타임이 길고 공급업체가 1곳이라 같은 결품 확률에서도 위험 지수가 높게 나온다.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(te["demand"], te["ma3"], s=8, alpha=0.5, label="MA3 baseline"); ax[0].scatter(te["demand"], te["ens"], s=8, alpha=0.5, label="ensemble")
mx = te["demand"].max(); ax[0].plot([0, mx], [0, mx], "k--", lw=1); ax[0].set_xlabel("actual"); ax[0].set_ylabel("forecast"); ax[0].legend()
ax[0].set_title("Hold-out actual vs forecast")
col = out["status"].map({"생산중": "tab:green", "단종예고": "tab:orange", "단종": "tab:red"})
ax[1].scatter(out["p_short"], out["risk"], c=col); ax[1].set_xlabel("shortage probability over lead time"); ax[1].set_ylabel("risk index")
ax[1].set_title("Stockout risk (green=active, orange=EOL notice, red=discontinued)")
plt.tight_layout(); plt.show()


## 실습 기록과 제출

마지막 **통합 산출물·학습기록 입력 화면**을 실행합니다. 각 산출물 탭에서 직접 내용을 작성하거나 기존 파일을 불러와 수정하고 개별 저장하세요. 학습기록 탭에서 실제 AI 질문·답변, 출력·자료 대조 근거, 검토 3건 이상 및 수정 내용, 최종 판단과 미달·미측정·미실시 상태를 기록합니다. 저장 후 ZIP 생성 및 다운로드 버튼을 순서대로 사용하세요.


**통합 입력·저장 기능:** 마지막 통합 화면에서 미션별 산출물을 파일별로 작성·수정·저장하고, 실제 AI 질문·답변 원문, 원자료·출력 대조 근거, 검토 3건, 수정과 최종 수행 상태를 기록한 뒤 ZIP으로 다운로드합니다. 저장만으로 PASS나 승인이 되지 않습니다.


In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
import json, csv, io, ast, zipfile, datetime, pathlib
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Markdown, FileLink, clear_output

MISSION_CODE = 5
DELIVERABLE_PATHS = ['forms/MNT02_M05_forms.md', 'reports/mnt02_model_metrics.md', 'src/models/mnt02_advanced_model.py']
PROJECT_NAME = 'MNT02'
NOTEBOOK_ROLE = '학생용 Student'
WORK_ROOT = Path.cwd().resolve()
if not globals().get('MNT02_PROJECT_READY', False):
    raise RuntimeError('먼저 ① 환경 설정 셀에서 실습자료 ZIP 파일을 업로드하고 완료를 확인하세요.')

RECORD_DIR = WORK_ROOT / f'MNT02_M{MISSION_CODE:02d}_학습기록'
RECORD_DIR.mkdir(parents=True, exist_ok=True)
ZIP_FILENAME = f'MNT02_m{MISSION_CODE:02d}_review.zip'

# 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
def safe_path(relative):
    target = (WORK_ROOT / relative).resolve()
    if not target.is_relative_to(WORK_ROOT):
        raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
    return target

def valid_text(path, content):
    suffix = Path(path).suffix.lower()
    if suffix == '.json':
        json.loads(content)
    elif suffix == '.ipynb':
        doc = json.loads(content)
        if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
            raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
    elif suffix == '.py':
        ast.parse(content)
    elif suffix == '.csv':
        if not content.strip():
            raise ValueError('CSV 데이터가 비어 있습니다.')
        rows = list(csv.reader(io.StringIO(content)))
        if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
            raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
        if any(len(row) != len(rows[0]) for row in rows if row):
            raise ValueError('CSV 열 개수가 행마다 다릅니다.')

# 산출물 입력 화면 — 파일별 별도 탭
editors = {}
file_tabs = []
file_buttons = {}
for relative in DELIVERABLE_PATHS:
    suffix = Path(relative).suffix.lower()
    hint = {
        '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
        '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
        '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
        '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
        '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
    }.get(suffix, '실제 작성한 내용을 입력합니다.')
    editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                              layout=widgets.Layout(width='100%', height='250px'))
    status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
    load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
    save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
    editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False}
    def on_edit(change, key=relative):
        state = editors[key]
        state['dirty'] = True
        state['saved'] = False
        state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
    editor.observe(on_edit, names='value')
    def load_clicked(button, key=relative):
        state = editors[key]
        target = safe_path(key)
        if not target.is_file():
            state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
            return
        if state['dirty']:
            state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
            return
        try:
            state['loaded'] = False
            state['editor'].value = target.read_text(encoding='utf-8')
            state['loaded'] = True
            state['dirty'] = False
            state['saved'] = True
            state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
        except Exception as exc:
            state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
    def save_clicked(button, key=relative):
        save_one(key)
    load_btn.on_click(load_clicked)
    save_btn.on_click(save_clicked)
    file_buttons[relative] = {'load':load_btn,'save':save_btn}
    file_tabs.append(widgets.VBox([
        widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
        editor, widgets.HBox([load_btn, save_btn]), status
    ]))

def save_one(relative):
    state = editors[relative]
    value = state['editor'].value
    if not value.strip():
        state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
        return False
    try:
        valid_text(relative, value)
        target = safe_path(relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(value, encoding='utf-8')
        (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
        state['loaded'] = True
        state['dirty'] = False
        state['saved'] = True
        state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
        return True
    except Exception as exc:
        state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
        return False

# 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
record_specs = [
 ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
 ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
 ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
 ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
 ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
 ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
]
record_widgets = {}
record_items = []
for key, title, hint in record_specs:
    field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
    record_widgets[key] = field
    record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

reviews = []
for i in range(3):
    claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
    ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
    action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
    reviews.append((claim,ground,action))
    record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                      value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                            value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
record_dirty = {'value': False}
def _mark_record_dirty(change):
    record_dirty['value'] = True
    record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
    field.observe(_mark_record_dirty,names='value')

# 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
previous_audit = RECORD_DIR/'learner_audit.json'
if previous_audit.is_file():
    try:
        previous = json.loads(previous_audit.read_text(encoding='utf-8'))
        for key, field in record_widgets.items():
            field.value = str(previous.get(key,'') or '')
        for group, past in zip(reviews, previous.get('reviews',[])):
            for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                widget.value = str(past.get(key,'') or '')
        status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
        if previous.get('instructor_approval') in approval.options:
            approval.value = previous['instructor_approval']
        record_dirty['value'] = False
        record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
    except Exception as exc:
        record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

def audit_object():
    return {
        'project': PROJECT_NAME, 'mission': MISSION_CODE, 'notebook_role': NOTEBOOK_ROLE,
        'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
        'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
        'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
        'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
        'status':list(status_check.value),'instructor_approval':approval.value,
        'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
    }

def save_audit(button=None):
    try:
        record = audit_object()
        (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
        (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
        sections = [f'# MNT02 미션 {MISSION_CODE:02d} 학습기록']
        for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
            sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
        for i,item in enumerate(record['reviews'],1):
            sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
        sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
        (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
        record_dirty['value'] = False
        (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
        completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
        record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
        return True
    except Exception as exc:
        record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
        return False

save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
save_audit_btn.on_click(save_audit)
record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

tabs = widgets.Tab(children=file_tabs+[record_tab])
for i, path in enumerate(DELIVERABLE_PATHS):
    tabs.set_title(i, f'산출물 {i+1}')
tabs.set_title(len(file_tabs),'학습기록')

summary_output = widgets.Output()
all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

def save_all(button):
    ok = 0
    for rel in DELIVERABLE_PATHS:
        if editors[rel]['editor'].value.strip() and save_one(rel):
            ok += 1
    with summary_output:
        summary_output.clear_output()
        print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

def build_zip(button):
    with summary_output:
        summary_output.clear_output()
        if record_dirty['value']:
            print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
            return
        dirty = [p for p,s in editors.items() if s['dirty'] and s['editor'].value.strip()]
        if dirty:
            print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
            for p in dirty: print(' -',p)
            return
        # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
        saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
        audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
        if not saved and not audit_files:
            print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
            return
        dest = WORK_ROOT / ZIP_FILENAME
        with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
            for rel in saved: archive.write(safe_path(rel), arcname=rel)
            for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
        print('✅ ZIP 생성:', dest)
        print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
        print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

def download_zip(button):
    with summary_output:
        summary_output.clear_output()
        target = WORK_ROOT / ZIP_FILENAME
        if record_dirty['value'] or any(v['dirty'] for v in editors.values()):
            print('미저장 변경이 있습니다. 먼저 저장 후 ZIP을 다시 만드세요.')
            return
        if not target.is_file():
            print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
            return
        try:
            from google.colab import files as colab_files
            colab_files.download(str(target))
            print('다운로드를 요청했습니다.')
        except ImportError:
            display(FileLink(str(target), result_html_prefix='ZIP 다운로드 링크: '))

all_btn.on_click(save_all)
zip_btn.on_click(build_zip)
dl_btn.on_click(download_zip)

print(f'✅ MNT02 미션 {MISSION_CODE:02d} ({NOTEBOOK_ROLE}) 산출물·학습기록 입력 화면')
display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
